## **Replicação de Artigos**

In [1]:
# Importando Bibliotecas
import os
import pandas as pd
import re
import functools

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Importando Modelos
from src.models.models_artigos.full_hrnet.full_hrnet import FullHRNet
from src.models.models_artigos.full_hrnet.full_hrnet_pretrained import FullHRNet_ImageNet
from src.models.models_artigos.resnet_point.two_stage_resnet import (GlobalStageWrapper,
                                                                     LocalStageWrapper,
                                                                     CroppedVFSSImageDataset,
                                                                     SwitchNorm2d,
                                                                     add_crop_boxes_to_df)
# RTMPose
from src.models.models_sota.rtmpose.rtmpose import RTMPose, download_rtmpose_backbone_weights

# Importar sua classe existente
from src.vfss_dataset import VFSSImageDataset

from src.split_data import split_data_k_fold
from src.utils import (plot_image_with_mask,
                       get_script_relative_path,
                       get_project_root_directory,
                       get_corners_from_angle,
                       clahe,
                       modify_input,
                       load_points,
                       custom_collate_fn,
                       set_seed,
                       avalia_status_GPU,
                       resolve_dataframe_path)

# Importando classes/funções
from src.training.config import TrainingConfig
from src.training.loss import (LossCalculator,
                               FocalMSELoss,
                               FocalMSEMaskedLoss)
from src.training.holdout import holdout
from src.training.cross_validation import cross_validate
from src.evalutation.inference import display_inference
from src.training.checkpoint_manager import CheckpointManager
from src.target.roi import generate_roi_from_points
from src.target.heatmap import generate_heatmap_from_points

set_seed(42)

## **RTMPose**

**Título do Artigo:** RTMPose: Real-Time Multi-Person Pose Estimation based on MMPose

### **Configuração**

In [ ]:
# ── Executar apenas se quiser pesos pré-treinados ──
# Requer: pip install timm

PRETRAINED_WEIGHTS_PATH = 'data/model_weights/rtmpose_cspnext_imagenet.pth'
os.makedirs('data/model_weights', exist_ok=True)

download_rtmpose_backbone_weights(save_path=PRETRAINED_WEIGHTS_PATH)

| Parâmetro | Do zero | Com pré-treinamento |
|-----------|---------|---------------------|
| `pretrained_path` | `None` | `PRETRAINED_WEIGHTS_PATH` |
| `learning_rate` | `3e-4` | `1e-4` (backbone já treinado) |
| `return_mode` | `"both"` | `"both"` |

**`width_mult`**: controla o tamanho do backbone  
- `0.5` → RTMPose-tiny (~4 M parâmetros)  
- `1.0` → RTMPose-small (~8 M parâmetros)

In [ ]:
# ── Escolha um dos dois blocos abaixo ──

# ── Opção A: Treinar do ZERO ──
rtmpose_config = TrainingConfig(
    path_dataframe = 'data/dataframes/video_frame.csv',

    model_class  = RTMPose,
    model_kwargs = dict(
        num_keypoints  = 2,
        in_channels    = 1,
        width_mult     = 0.5,        # tiny
        output_size    = (256, 256),
        pretrained_path = None,      # sem pré-treinamento
        return_mode    = 'both',
    ),

    test_size    = 0.2,
    n_folds      = 5,
    dataset_class = VFSSImageDataset,
    sigma_heatmap = 20,

    epochs        = 200,
    learning_rate = 3e-4,
    batch_size    = 8,
    patience      = 5,

    predict_roi       = True,
    weight_roi        = 0.4,
    weight_heatmap    = 0.4,
    weight_penalty    = 0.2,
    weight_keypoints  = 0.0,
)

print(f"Experimento: {rtmpose_config.id_experiment}")
print(f"Checkpoint:  {rtmpose_config.checkpoint_dir}")

In [ ]:
# ── Opção B: Treinar com pré-treinamento ImageNet ──
# (Execute este bloco *em vez* do anterior)

rtmpose_config = TrainingConfig(
    path_dataframe = 'data/dataframes/video_frame.csv',

    model_class  = RTMPose,
    model_kwargs = dict(
        num_keypoints   = 2,
        in_channels     = 1,
        width_mult      = 0.5,
        output_size     = (256, 256),
        pretrained_path = PRETRAINED_WEIGHTS_PATH,  # pesos baixados acima
        return_mode     = 'both',
    ),

    test_size    = 0.2,
    n_folds      = 5,
    dataset_class = VFSSImageDataset,
    sigma_heatmap = 20,

    epochs        = 200,
    learning_rate = 1e-4,   # LR menor para fine-tuning
    batch_size    = 8,
    patience      = 5,

    predict_roi       = True,
    weight_roi        = 0.4,
    weight_heatmap    = 0.4,
    weight_penalty    = 0.2,
    weight_keypoints  = 0.0,
)

print(f"Experimento: {rtmpose_config.id_experiment}")
print(f"Checkpoint:  {rtmpose_config.checkpoint_dir}")

In [ ]:
# Sanity check — verificar saída do modelo
avalia_status_GPU()

model_test = RTMPose(**rtmpose_config.model_kwargs)
x_dummy = torch.zeros(1, 1, 256, 256)
roi_out, hm_out, _ = model_test(x_dummy)

print(f"ROI shape:     {roi_out.shape if roi_out is not None else 'None'}")
print(f"Heatmap shape: {hm_out.shape  if hm_out  is not None else 'None'}")
print(f"Parâmetros:    {sum(p.numel() for p in model_test.parameters()):,}")
del model_test

### **Treino/Validação**

In [ ]:
cross_validate(rtmpose_config)

### **Teste**

In [ ]:
display_inference(rtmpose_config)